# Analiza wyników uczniów - projekt ze Statystyki i Teorii Obsługi Masowej

Ten notebook jest głównym raportem projektu. Zawiera część statystyczną, analizę regresji wielorakiej, diagnostykę reszt oraz późniejszą część z Teorii Obsługi Masowej i symulacji EDS.

In [ ]:
from analysis import (
    load_student_performance_data,
    summarize_simulation_results,
    run_simulation_experiments,
    plot_mean_waiting_time_histogram,
    summarize_data,
    run_all_hypothesis_tests,
    run_regression_analysis,
    run_extended_regression_analysis,
    test_teacher_parent_vs_final_grade,
    test_parent_status_vs_family_relationship,
    test_parent_education_vs_higher_education_plan,
    test_parent_education_vs_study_time_for_good_students,
)


## 1. Cel projektu

W części statystycznej wykorzystujemy zbiór UCI Student Performance. Celem jest sprawdzenie wybranych hipotez statystycznych oraz zbudowanie modelu regresji wielorakiej wyjaśniającego ocenę końcową `G3`.

W części TOM zostanie zamodelowane wybrane zjawisko kolejkowe związane z koleją i przeprowadzona symulacja zdarzeń dyskretnych.

## 2. Wczytanie i podstawowy opis danych

W projekcie wykorzystujemy zbiór Student Performance z repozytorium UCI. Dane opisują wyniki uczniów dwóch portugalskich szkół średnich. Informacje pochodzą z raportów szkolnych oraz ankiet.

Każdy wiersz odpowiada jednemu uczniowi. W danych znajdują się zmienne opisujące sytuację rodzinną, warunki nauki, styl życia, wsparcie edukacyjne oraz oceny ucznia.

Najważniejszą zmienną w projekcie jest `G3`, czyli ocena końcowa ucznia w skali od 0 do 20. Zmienne `G1` i `G2` oznaczają wcześniejsze oceny okresowe. W regresji nie używamy ich jako predyktorów, ponieważ są bardzo bezpośrednio powiązane z oceną końcową i mogłyby sztucznie zawyżyć jakość modelu.

Dane można podzielić na kilka grup:

- zmienne demograficzne i rodzinne, np. `age`, `sex`, `Pstatus`, `Medu`, `Fedu`, `Mjob`, `Fjob`,
- zmienne szkolne i edukacyjne, np. `studytime`, `failures`, `schoolsup`, `higher`, `internet`,
- zmienne społeczne i stylu życia, np. `famrel`, `freetime`, `goout`, `health`, `absences`,
- zmienne ocen: `G1`, `G2`, `G3`.

In [ ]:
features, targets, data = load_student_performance_data()

print("Features:", features.shape)
print("Targets:", targets.shape)
print("Full data:", data.shape)

data.head()


In [ ]:
describe_summary, value_summary = summarize_data(data)

display(describe_summary)
display(value_summary)


## 3. Hipotezy statystyczne

W tej części sprawdzamy cztery hipotezy dotyczące uczniów, ich sytuacji rodzinnej, planów edukacyjnych oraz ocen końcowych. Dla każdej hipotezy podajemy pytanie badawcze, dobrany test statystyczny, najważniejsze założenia oraz interpretację wyniku przy poziomie istotności alfa = 0.05.

### Hipotezy

1. Uczniowie, których co najmniej jeden rodzic pracuje jako nauczyciel, osiągają wyższe oceny końcowe niż uczniowie bez rodzica nauczyciela.
2. Uczniowie, których rodzice mieszkają osobno, deklarują słabsze relacje rodzinne niż uczniowie, których rodzice mieszkają razem.
3. Uczniowie, których co najmniej jeden rodzic ma wyższe wykształcenie, częściej planują podjęcie studiów.
4. Wśród uczniów z dobrymi ocenami końcowymi osoby mające co najmniej jednego wysoko wykształconego rodzica deklarują krótszy czas nauki niż pozostali uczniowie.

### Dobór testów i uzasadnienie

- Hipoteza 1 wykorzystuje test t dla dwóch prób niezależnych w wersji Welcha, ponieważ ocena końcowa `G3` jest zmienną liczbową, a porównujemy dwie niezależne grupy. Wersja Welcha jest bezpieczniejsza przy nierównych liczebnościach i wariancjach grup.
- Hipoteza 2 wykorzystuje test U Manna-Whitneya, ponieważ `famrel` jest zmienną porządkową od 1 do 5, a porównywane grupy są niezależne.
- Hipoteza 3 wykorzystuje test chi-kwadrat niezależności, ponieważ obie zmienne są kategoryczne: wysoko wykształcony rodzic tak/nie oraz plan podjęcia studiów tak/nie.
- Hipoteza 4 wykorzystuje test U Manna-Whitneya, ponieważ `studytime` jest zmienną porządkową od 1 do 4, a porównujemy dwie niezależne grupy uczniów z dobrymi ocenami.

Najważniejsze założenia do omówienia na prezentacji to: niezależność obserwacji, odpowiednia skala pomiarowa dla danego testu, przybliżona normalność dla testu t lub uzasadnienie użycia wersji Welcha oraz oczekiwane liczebności w teście chi-kwadrat.

In [ ]:
test_teacher_parent_vs_final_grade(data)


In [ ]:
test_parent_status_vs_family_relationship(data)


In [ ]:
test_parent_education_vs_higher_education_plan(data)


In [ ]:
test_parent_education_vs_study_time_for_good_students(data)


## 4. Regresja wieloraka

W tej części porównujemy dwa modele przewidujące ocenę końcową `G3`. Zmienne `G1` i `G2` są pominięte celowo, ponieważ są wcześniejszymi ocenami i zbyt mocno wyjaśniałyby wynik końcowy. Zmienne kategoryczne są kodowane w formule przez `C(zmienna)`.

### 4.1 Model bazowy

Model bazowy wykorzystuje najważniejsze zmienne edukacyjne i rodzinne: czas nauki, wcześniejsze niezaliczenia, nieobecności, wykształcenie rodziców, czas dojazdu oraz kilka zmiennych `yes/no`.


In [ ]:
run_regression_analysis(data)


Model bazowy jest istotny jako całość i wyjaśnia około 26% zmienności oceny końcowej `G3`. Najważniejsze istotne zmienne to `studytime`, `failures`, `schoolsup` oraz `higher`. VIF-y są niskie, więc nie widać problemu silnej współliniowości. Diagnostyka reszt wskazuje jednak na problem z homoskedastycznością.


### 4.2 Model rozszerzony

Model rozszerzony dodaje więcej zmiennych opisujących wsparcie edukacyjne, aktywności oraz styl życia ucznia. Służy do sprawdzenia, czy szerszy zestaw zmiennych poprawia dopasowanie modelu i czy nadal nie pojawia się problem współliniowości.


In [ ]:
run_extended_regression_analysis(data)


Model rozszerzony ma nieco lepsze dopasowanie niż model bazowy: `R2` i skorygowane `R2` są wyższe, a `AIC` niższe. Dodatkowo istotna okazała się zmienna `Dalc`, czyli spożycie alkoholu w dni robocze. `BIC` jest jednak wyższe niż w modelu bazowym, więc prostszy model nadal ma przewagę pod względem oszczędności liczby zmiennych.


### 4.3 Porównanie modeli

Model bazowy jest prostszy i łatwiejszy do interpretacji. Model rozszerzony trochę poprawia dopasowanie, ale nie rozwiązuje problemów diagnostycznych reszt. Do prezentacji można potraktować model bazowy jako główny, a model rozszerzony jako sprawdzenie, czy dodanie zmiennych społecznych i stylu życia poprawia wynik.


## 5. Teoria Obsługi Masowej / symulacja EDS

W części TOM modelujemy kolejkę pasażerów do jednego biletomatu na stacji kolejowej. Jest to osobny model symulacyjny, niezależny od danych uczniów używanych w części statystycznej. Celem jest sprawdzenie, jak długo pasażerowie przeciętnie czekają na obsługę oraz jak mocno wykorzystywany jest biletomat.

### 5.1 Opis modelowanego zjawiska

Pasażerowie przychodzą na stację i chcą kupić bilet w biletomacie. Jeśli biletomat jest wolny, pasażer zaczyna obsługę od razu. Jeśli biletomat jest zajęty, pasażer trafia do kolejki. Kolejka działa zgodnie z zasadą FIFO, czyli pierwszy przychodzi i pierwszy jest obsłużony.


### 5.2 Założenia modelu

- system ma jedno stanowisko obsługi, czyli jeden biletomat,
- pasażerowie są obsługiwani w kolejności FIFO,
- czas symulacji to 8 godzin, czyli 480 minut,
- odstępy między przyjściami pasażerów mają rozkład wykładniczy,
- czas obsługi ma rozkład trójkątny,
- analizujemy głównie średni czas oczekiwania pasażera.

Rozkład wykładniczy jest użyty dla przyjść, ponieważ dobrze opisuje losowe odstępy między niezależnymi przyjściami pasażerów. Rozkład trójkątny jest użyty dla czasu obsługi, ponieważ łatwo określić minimalny, typowy i maksymalny czas zakupu biletu.


### 5.3 Implementacja i parametry symulacji

Symulacja jest typu EDS, czyli zdarzeń dyskretnych. Nie przechodzimy minuta po minucie, tylko przeskakujemy pomiędzy zdarzeniami: przyjściem pasażera albo zakończeniem obsługi.


In [ ]:
simulation_time = 480
repetitions = 100
mean_interarrival_time = 2.5

service_min = 0.5
service_mode = 1.5
service_max = 4.0

simulation_results = run_simulation_experiments(
    repetitions=repetitions,
    simulation_time=simulation_time,
    mean_interarrival_time=mean_interarrival_time,
    service_min=service_min,
    service_mode=service_mode,
    service_max=service_max,
)

display(simulation_results.head())


### 5.4 Wyniki symulacji

Symulację powtarzamy 100 razy, żeby nie opierać wniosku na pojedynczym losowym przebiegu. Następnie opisujemy podstawowe charakterystyki wyników.

Z uzyskanych wyników wynika, że w jednym 8-godzinnym przebiegu obsługiwano średnio około 189 pasażerów. Średni czas oczekiwania wyniósł 4.027 min, średnia długość kolejki 1.633 osoby, a wykorzystanie biletomatu 0.789, czyli około 79% czasu pracy. Maksymalna długość kolejki wynosiła przeciętnie 8.36 osoby, ale w najbardziej obciążonym przebiegu doszła do 19 osób.


In [ ]:
simulation_summary = summarize_simulation_results(simulation_results)
display(simulation_summary)


### 5.5 Analiza wybranej charakterystyki

Jako główną charakterystykę wybieramy średni czas oczekiwania pasażera na rozpoczęcie obsługi. Jest to praktyczna miara jakości działania systemu z punktu widzenia pasażera.

W 100 powtórzeniach średni czas oczekiwania wyniósł 4.027 min, przy odchyleniu standardowym 1.792 min. Najkrótszy średni czas oczekiwania w pojedynczym przebiegu wyniósł 1.317 min, a najdłuższy 10.752 min. Histogram pokazuje, że większość symulacji skupia się w okolicach kilku minut oczekiwania, ale zdarzają się też pojedyncze przebiegi z wyraźnie dłuższym oczekiwaniem.


In [ ]:
mean_waiting_time = simulation_results["mean_waiting_time"]

print(f"Średni czas oczekiwania: {mean_waiting_time.mean():.3f} min")
print(f"Odchylenie standardowe: {mean_waiting_time.std():.3f} min")
print(f"Minimum: {mean_waiting_time.min():.3f} min")
print(f"Maksimum: {mean_waiting_time.max():.3f} min")

plot_mean_waiting_time_histogram(simulation_results)


### 5.6 Wniosek z części TOM

Dla przyjętych parametrów pasażer czekał średnio około 4.027 minuty na rozpoczęcie obsługi, a biletomat był zajęty przez około 78.9% czasu. Średnia długość kolejki wyniosła około 1.633 osoby, więc system działał stabilnie, ale był dość mocno obciążony.

Wniosek praktyczny jest taki, że jeden biletomat wystarcza w scenariuszu bazowym, ale nie ma bardzo dużego zapasu przepustowości. Przy większym natężeniu pasażerów albo dłuższym czasie obsługi kolejka mogłaby szybko rosnąć, dlatego w godzinach szczytu warto rozważyć drugi biletomat albo dodatkowy kanał sprzedaży biletów.


## 6. Wnioski końcowe

W części statystycznej potwierdzono dwie z czterech hipotez. Istotne okazało się to, że uczniowie z co najmniej jednym rodzicem nauczycielem osiągali wyższe oceny końcowe oraz że wyższe wykształcenie rodzica było powiązane z planem podjęcia studiów. Nie potwierdzono statystycznie hipotez dotyczących słabszych relacji rodzinnych przy rodzicach mieszkających osobno oraz krótszego czasu nauki wśród dobrych uczniów z wysoko wykształconym rodzicem.

W regresji oba modele były istotne jako całość. Model rozszerzony miał trochę lepsze dopasowanie niż bazowy, ale był bardziej złożony. Najważniejsze zmienne powiązane z oceną końcową to czas nauki, wcześniejsze niezaliczenia, dodatkowe wsparcie szkolne, plan podjęcia studiów oraz w modelu rozszerzonym spożycie alkoholu w dni robocze. Diagnostyka reszt wskazała jednak na problem heteroskedastyczności, więc modele należy traktować głównie opisowo.

W części TOM zasymulowano kolejkę pasażerów do jednego biletomatu na stacji kolejowej. Dla przyjętych parametrów średni czas oczekiwania wyniósł około 4.027 minuty, a wykorzystanie biletomatu około 78.9%. System działał stabilnie, ale był dość mocno obciążony, dlatego przy większym ruchu warto rozważyć dodatkowy biletomat.
